# 02 — Data Profiling and Grain Analysis

**Project Owner & Lead Analyst:** Abdeljalil El Khyati

**Project:** OULAD — EdTech Learning Analytics  

## Objective
Understand the actual data structure, grain, and basic statistics before making quality decisions or designing the PostgreSQL RAW schema.

## Analyst Question
**Question:** What do we need to know before joining tables or writing SQL transformations?  
**Answer:** We need to identify the table grain, row counts, key behavior, missingness, value distributions, and relationships.

## The Golden Rule
**Important:** Repeated `id_student` is not automatically a duplicate. A student can legitimately have many activity records in `studentVle` or multiple registrations. We must define what constitutes a true duplicate based on the Grain.

## Profiling strategy: SQL-first with Python as a complement

PostgreSQL is used for relational counts, key checks, and joins once the data is loaded. Python supports source-level profiling before ingestion and targeted analysis that benefits from its libraries.

The profiling record distinguishes observed structure from assumptions. Repeated identifiers are evaluated against the table grain, and each reported measure is tied to a defined technical or business question.

## Interpretation framework

For each profiling result, the analysis records the expectation, observed value, compatibility with the table grain, effect on the next decision, and any unresolved branch.

In [1]:
import os
import pandas as pd
from pathlib import Path

# Ensure we are pointing to the correct raw directory using robust path resolution
current_dir = Path.cwd().resolve()
if current_dir.name == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

FILES_TO_PROFILE = [
    "courses.csv",
    "assessments.csv",
    "studentInfo.csv",
    "studentRegistration.csv",
    "studentAssessment.csv",
    "vle.csv",
    "studentVle.csv"
]

print("--- PRE-INGESTION DATA PROFILING ---\n")

for file_name in FILES_TO_PROFILE:
    file_path = RAW_DATA_DIR / file_name
    
    if not file_path.exists():
        print(f"[ERROR] {file_name} not found. Run ingestion first.")
        continue
        
    print(f"=== {file_name} ===")
    
    # 1. Read ONLY the first 5 rows to understand structure safely (Memory Efficient)
    df_sample = pd.read_csv(file_path, nrows=5)
    
    # 2. Get total row count efficiently without loading the whole file into RAM
    with open(file_path, 'r', encoding='utf-8') as f:
        total_rows = sum(1 for _ in f) - 1 # Subtract 1 for the header row
    
    print(f"Total Rows: {total_rows:,}")
    print(f"Total Columns: {len(df_sample.columns)}")
    print("\nColumns & Data Types:")
    print(df_sample.dtypes.to_string())
    
    print("\nSample Data (First 3 rows):")
    display(df_sample.head(3))
    print("\n" + "="*60 + "\n")

--- PRE-INGESTION DATA PROFILING ---

=== courses.csv ===
Total Rows: 22
Total Columns: 3

Columns & Data Types:
code_module                     str
code_presentation               str
module_presentation_length    int64

Sample Data (First 3 rows):


,code_module,code_presentation,module_presentation_length
0,AAA,2013J,268
1,AAA,2014J,269
2,BBB,2013J,268




=== assessments.csv ===
Total Rows: 206
Total Columns: 6

Columns & Data Types:
code_module            str
code_presentation      str
id_assessment        int64
assessment_type        str
date                 int64
weight               int64

Sample Data (First 3 rows):


,code_module,code_presentation,id_assessment,assessment_type,date,weight
0,AAA,2013J,1752,TMA,19,10
1,AAA,2013J,1753,TMA,54,20
2,AAA,2013J,1754,TMA,117,20




=== studentInfo.csv ===
Total Rows: 32,593
Total Columns: 12

Columns & Data Types:
code_module               str
code_presentation         str
id_student              int64
gender                    str
region                    str
highest_education         str
imd_band                  str
age_band                  str
num_of_prev_attempts    int64
studied_credits         int64
disability                str
final_result              str

Sample Data (First 3 rows):


...

,id_assessment,id_student,date_submitted,is_banked,score
0,1752,11391,18,0,78
1,1752,28400,22,0,70
2,1752,31604,17,0,72




=== vle.csv ===
Total Rows: 6,364
Total Columns: 6

Columns & Data Types:
id_site              int64
code_module            str
code_presentation      str
activity_type          str
week_from              str
week_to                str

Sample Data (First 3 rows):


,id_site,code_module,code_presentation,activity_type,week_from,week_to
0,546943,AAA,2013J,resource,?,?
1,546712,AAA,2013J,oucontent,?,?
2,546998,AAA,2013J,resource,?,?




=== studentVle.csv ===
Total Rows: 10,655,280
Total Columns: 6

Columns & Data Types:
code_module            str
code_presentation      str
id_student           int64
id_site              int64
date                 int64
sum_click            int64

Sample Data (First 3 rows):


,code_module,code_presentation,id_student,id_site,date,sum_click
0,AAA,2013J,28400,546652,-10,4
1,AAA,2013J,28400,546652,-10,1
2,AAA,2013J,28400,546652,-10,1


## 3. Profiling Results Summary

The profiling output reveals the following structure:

### Table Structure Overview

| Table | Rows | Grain | Primary Key Candidate | Key Observations |
|-------|------|-------|----------------------|------------------|
| **courses** | 22 | One module presentation | `(code_module, code_presentation)` | Small dimension table |
| **assessments** | 206 | One assessment per presentation | `id_assessment` | `date` is int64 (days from start) |
| **studentInfo** | 32,593 | One student per presentation | `(code_module, code_presentation, id_student)` | Contains `final_result` (target variable) |
| **studentRegistration** | 32,593 | One registration event | `(code_module, code_presentation, id_student)` | `date_unregistration` has "?" for non-withdrawn |
| **studentAssessment** | 173,912 | One submission per student per assessment | `(id_assessment, id_student)` | `date_submitted` is int64 |
| **vle** | 6,364 | One VLE resource per presentation | `id_site` | `week_from`/`week_to` have "?" |
| **studentVle** | **10,655,280** | One student's clicks on one resource on one day | `(code_module, code_presentation, id_student, id_site, date)` | Largest table, `date` can be negative |

### Critical Observations

1. **Dates as Integers:** All dates are stored as `int64` representing days from presentation start (day 0 = start). Negative values mean "before start".
2. **Missing Values as "?":** Some columns use `"?"` instead of `NULL` (e.g., `date_unregistration`, `week_from`). This will require handling in the Data Quality stage.
3. **Composite Grain:** `id_student` is not unique per person, but per (student + module + presentation). The same student may appear multiple times across different enrollments.

### Next Step

These observations will guide:
- PostgreSQL `CREATE TABLE` data type decisions (e.g., `INTEGER` for dates, `TEXT` for "?" placeholders)
- Primary key constraints in the RAW layer
- Data Quality checks for "?" values and date ranges

## Profiling outcomes and schema decisions

The profiling evidence informed `docs/data_dictionary.md` and the RAW table definitions in `sql/01_raw/`. The RAW schema has been created and loaded. The subsequent engineering phase characterized repeated `studentVle` keys without deleting or aggregating source rows.

### 2.1 Structural data contract

The structural data contract is based on the observed CSV headers, column order, missing-value markers, and value types. Source structure is inspected directly rather than inferred solely from external documentation.

**Expected evidence:** Exact column names and order can be read from each CSV header without loading the full dataset into memory.

**Method rationale:** Reading only the header with `csv.reader` provides the schema evidence needed for RAW design while avoiding unnecessary memory use for the large `studentVle.csv` file.

**Design rule:** RAW preserves source column names and source representations. Semantic conversion is deferred to STAGING.

In [2]:
from pathlib import Path
import csv

print("--- STRUCTURAL PROFILE ---\n")

for file_name in FILES_TO_PROFILE:
    file_path = RAW_DATA_DIR / file_name
    
    print(f"=== {file_name} ===")
    
    # Read header directly from the CSV
    with open(file_path, "r", encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        header = next(reader)
        
    print("Columns:")
    for i, column in enumerate(header, start=1):
        print(f"  {i}. {column}")
        
    print(f"Column count: {len(header)}\n")

--- STRUCTURAL PROFILE ---

=== courses.csv ===
Columns:
  1. code_module
  2. code_presentation
  3. module_presentation_length
Column count: 3

=== assessments.csv ===
Columns:
  1. code_module
  2. code_presentation
  3. id_assessment
  4. assessment_type
  5. date
  6. weight
Column count: 6

=== studentInfo.csv ===
Columns:
  1. code_module
  2. code_presentation
  3. id_student
  4. gender
  5. region
  6. highest_education
  7. imd_band
  8. age_band
  9. num_of_prev_attempts
  10. studied_credits
  11. disability
  12. final_result
Column count: 12

=== studentRegistration.csv ===
Columns:
  1. code_module
  2. code_presentation
  3. id_student
  4. date_registration
  5. date_unregistration
Column count: 5

=== studentAssessment.csv ===
Columns:
  1. id_assessment
  2. id_student
  3. date_submitted
  4. is_banked
  5. score
Column count: 5

=== vle.csv ===
Columns:
  1. id_site
  2. code_module
  3. code_presentation
  4. activity_type
  5. week_from
  6. week_to
Column count

### 2.2 Structural validation and findings

The structural profile confirmed the source column names and order. It also showed that `date_unregistration`, `week_from`, and `week_to` contain the source placeholder `?`, so they were read as text at source level.

**Decision:** Preserve the observed source representation in RAW and defer type conversion to STAGING, where the documented missing-token mapping can be validated.

In [3]:
import pandas as pd

print("--- CANDIDATE KEY VALIDATION & NULL CHECK ---\n")

# Define candidate keys for validation
candidate_keys = {
    "studentInfo.csv": ["code_module", "code_presentation", "id_student"],
    "studentRegistration.csv": ["code_module", "code_presentation", "id_student"],
    "studentAssessment.csv": ["id_assessment", "id_student"],
    "studentVle.csv": ["code_module", "code_presentation", "id_student", "id_site", "date"]
}

for file_name, key_cols in candidate_keys.items():
    file_path = RAW_DATA_DIR / file_name
    
    # Memory-efficient read: load ONLY the candidate key columns
    df_keys = pd.read_csv(file_path, usecols=key_cols)
    
    total_rows = len(df_keys)
    unique_rows = df_keys.drop_duplicates().shape[0]
    is_unique = (total_rows == unique_rows)
    null_counts = df_keys.isnull().sum()
    has_nulls_in_key = null_counts.sum() > 0
    
    print(f"=== {file_name} ===")
    print(f"Candidate Key: {key_cols}")
    print(f"Total Rows: {total_rows:,} | Unique Rows: {unique_rows:,}")
    print(f"Is Unique (COUNT(*) == COUNT(DISTINCT))? : {'YES' if is_unique else 'NO'}")
    print(f"Contains NULLs in Key Columns? : {'YES' if has_nulls_in_key else 'NO'}")
    if has_nulls_in_key:
        print(f"Null counts per column:\n{null_counts[null_counts > 0]}")
    print("-" * 50)

--- CANDIDATE KEY VALIDATION & NULL CHECK ---

=== studentInfo.csv ===
Candidate Key: ['code_module', 'code_presentation', 'id_student']
Total Rows: 32,593 | Unique Rows: 32,593
Is Unique (COUNT(*) == COUNT(DISTINCT))? : YES
Contains NULLs in Key Columns? : NO
--------------------------------------------------
=== studentRegistration.csv ===
Candidate Key: ['code_module', 'code_presentation', 'id_student']
Total Rows: 32,593 | Unique Rows: 32,593
Is Unique (COUNT(*) == COUNT(DISTINCT))? : YES
Contains NULLs in Key Columns? : NO
--------------------------------------------------
=== studentAssessment.csv ===
Candidate Key: ['id_assessment', 'id_student']
Total Rows: 173,912 | Unique Rows: 173,912
Is Unique (COUNT(*) == COUNT(DISTINCT))? : YES
Contains NULLs in Key Columns? : NO
--------------------------------------------------
=== studentVle.csv ===
Candidate Key: ['code_module', 'code_presentation', 'id_student', 'id_site', 'date']
Total Rows: 10,655,280 | Unique Rows: 8,459,320
Is Un

### Candidate-key validation interpretation

A candidate key is valid only when its components are non-null and the complete key is unique at the declared grain. Equal row and distinct-key counts support uniqueness; separate null checks establish key completeness.

A key that fails either condition is not suitable for a primary-key constraint without further investigation or a revised grain definition.

# Phase 2A — RAW Type and Null Validation

## Objective

Before creating PostgreSQL RAW tables, validate:
- actual values and types of `weight` and `score`
- missing values and `?` markers across all seven source CSV files
- source column names and column order

The goal is to design RAW tables from observed source data, not assumptions.

In [4]:
import pandas as pd
from pathlib import Path

# Resolve project root
current_dir = Path.cwd().resolve()

if current_dir.name == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DATA_DIR}")

Project root: G:\OULAD-EdTech-Learning-Analytics-starter\OULAD-EdTech-Learning-Analytics
Raw data directory: G:\OULAD-EdTech-Learning-Analytics-starter\OULAD-EdTech-Learning-Analytics\data\raw


## 1. `assessments.csv` — Validate `weight`

We need to know whether `weight` contains only integer-valued numbers or real decimal values.

We also check missing values and the actual range of values.

In [5]:
df_assessments = pd.read_csv(RAW_DATA_DIR / "assessments.csv")

weight = df_assessments["weight"]

print("=== assessments.csv / weight ===")
print(f"dtype: {weight.dtype}")
print(f"Missing values: {weight.isna().sum()}")
print(f"Unique values: {sorted(weight.dropna().unique().tolist())}")
print(f"Min: {weight.min()}")
print(f"Max: {weight.max()}")

integer_valued = (weight.dropna() % 1 == 0).all()

print(f"All non-missing values are integer-valued: {integer_valued}")

=== assessments.csv / weight ===
dtype: float64
Missing values: 0
Unique values: [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 7.5, 8.0, 9.0, 10.0, 12.5, 15.0, 16.0, 17.5, 18.0, 20.0, 22.0, 25.0, 28.0, 30.0, 35.0, 100.0]
Min: 0.0
Max: 100.0
All non-missing values are integer-valued: False


## Interpretation

If all non-missing values are integer-valued, `INTEGER` is acceptable.

If real fractional values exist, use `NUMERIC`.

## 2. `studentAssessment.csv` — Validate `score`

We apply the same validation to assessment scores.

In [7]:
df_student_assessment = pd.read_csv(
    RAW_DATA_DIR / "studentAssessment.csv",
    keep_default_na=False # Prevent pandas from auto-converting "?" to NaN prematurely
)

score = df_student_assessment["score"]

print("=== studentAssessment.csv / score ===")
print(f"Raw dtype: {score.dtype}")

# Count specific missing markers
question_mark_count = (score == "?").sum()
print(f"'?' placeholders: {question_mark_count}")

# Convert to numeric, coercing "?" and other non-numeric strings to NaN
score_numeric = pd.to_numeric(score, errors='coerce')

print(f"Numeric Missing values (NaN): {score_numeric.isna().sum()}")
print(f"Min (numeric): {score_numeric.min()}")
print(f"Max (numeric): {score_numeric.max()}")

unique_scores_numeric = sorted(score_numeric.dropna().unique().tolist())
print(f"Number of unique numeric values: {len(unique_scores_numeric)}")
print(f"First 30 unique numeric values: {unique_scores_numeric[:30]}")

# Now it is safe to check for integer values
integer_valued = (score_numeric.dropna() % 1 == 0).all()

print(f"All non-missing numeric values are integer-valued: {integer_valued}")

if question_mark_count > 0:
    print("\n⚠️ DECISION: Column contains '?' placeholders. Must be TEXT in RAW layer.")
else:
    print("\n✅ DECISION: Column is clean. Can use NUMERIC or INTEGER based on decimal check.")

=== studentAssessment.csv / score ===
Raw dtype: str
'?' placeholders: 173
Numeric Missing values (NaN): 173
Min (numeric): 0.0
Max (numeric): 100.0
Number of unique numeric values: 101
First 30 unique numeric values: [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0]
All non-missing numeric values are integer-valued: True

⚠️ DECISION: Column contains '?' placeholders. Must be TEXT in RAW layer.


## Interpretation for `score`

1. **Source Reality:** The column contains the `"?"` placeholder for missing scores.
2. **RAW Layer Decision:** Must be defined as `TEXT` in PostgreSQL to preserve source fidelity. We do not replace `"?"` with `NULL` in RAW.
3. **STAGING Layer Plan:** In the staging phase, we will replace `"?"` with `NULL` and cast the column to `NUMERIC` or `INTEGER` (since the valid values are integers).

# Phase 2A — Full Source Missing-Value Audit

## Objective

RAW must preserve the source faithfully.

Before defining `NOT NULL` constraints, we inspect every column in every source CSV for:
- missing values
- `?` markers
- empty strings

In [8]:
# ============================================================================
# OPTION A — Director's Solution (Loads all tables into memory)
# ============================================================================
source_files = {
    "courses": "courses.csv",
    "assessments": "assessments.csv",
    "student_info": "studentInfo.csv",
    "student_registration": "studentRegistration.csv",
    "student_assessment": "studentAssessment.csv",
    "vle": "vle.csv",
    "student_vle": "studentVle.csv",
}

for table_name, filename in source_files.items():
    print("\n" + "=" * 80)
    print(f"{table_name} -> {filename}")
    print("=" * 80)

    df = pd.read_csv(
        RAW_DATA_DIR / filename,
        keep_default_na=False
    )

    for column in df.columns:
        missing_count = (
            df[column].isna().sum()
            + (df[column] == "").sum()
            + (df[column] == "?").sum()
        )

        if missing_count > 0:
            print(f"{column}: {missing_count} missing/empty/? values")

# ============================================================================
# OPTION B — Memory-Efficient Streaming (Alternative for 8 GB RAM devices)
# To activate: Remove the '#' from each line below and comment out OPTION A
# ============================================================================
# import csv
#
# # Small tables: use pandas (fast)
# small_tables = {
#     "courses": "courses.csv",
#     "assessments": "assessments.csv",
#     "student_info": "studentInfo.csv",
#     "student_registration": "studentRegistration.csv",
#     "student_assessment": "studentAssessment.csv",
#     "vle": "vle.csv",
# }
#
# for table_name, filename in small_tables.items():
#     print("\n" + "=" * 80)
#     print(f"{table_name} -> {filename}")
#     print("=" * 80)
#
#     df = pd.read_csv(RAW_DATA_DIR / filename, keep_default_na=False)
#
#     for column in df.columns:
#         missing_count = (
#             df[column].isna().sum()
#             + (df[column] == "").sum()
#             + (df[column] == "?").sum()
#         )
#         if missing_count > 0:
#             print(f"{column}: {missing_count} missing/empty/? values")
#
# # Large table: use streaming (memory-efficient)
# print("\n" + "=" * 80)
# print("student_vle -> studentVle.csv (STREAMING MODE)")
# print("=" * 80)
#
# with open(RAW_DATA_DIR / "studentVle.csv", 'r', encoding='utf-8') as f:
#     reader = csv.reader(f)
#     columns = next(reader)
#
#     missing_counts = {col: 0 for col in columns}
#     row_count = 0
#
#     for row in reader:
#         row_count += 1
#         for i, value in enumerate(row):
#             if value == '' or value == '?':
#                 missing_counts[columns[i]] += 1
#
#         if row_count % 2_000_000 == 0:
#             print(f"  ... processed {row_count:,} rows")
#
# print(f"\nTotal rows: {row_count:,}")
# for col, count in missing_counts.items():
#     if count > 0:
#         print(f"{col}: {count} missing/empty/? values")
#
# if all(c == 0 for c in missing_counts.values()):
#     print("✅ No missing/empty/? values detected in studentVle.csv")


courses -> courses.csv

assessments -> assessments.csv
date: 11 missing/empty/? values

student_info -> studentInfo.csv
imd_band: 1111 missing/empty/? values

student_registration -> studentRegistration.csv
date_registration: 45 missing/empty/? values
date_unregistration: 22521 missing/empty/? values

student_assessment -> studentAssessment.csv
score: 173 missing/empty/? values

vle -> vle.csv
week_from: 5243 missing/empty/? values
week_to: 5243 missing/empty/? values

student_vle -> studentVle.csv


## Interpretation

Only columns with zero missing/empty/? values should be considered for `NOT NULL`.

Columns containing source missing markers must remain nullable in RAW unless the source representation is deliberately preserved as text.

# Source Column Order Validation

## Objective

Before PostgreSQL COPY, verify that the CSV headers match the expected RAW table columns.

In [9]:
expected_columns = {
    "courses.csv": [
        "code_module",
        "code_presentation",
        "module_presentation_length",
    ],
    "assessments.csv": [
        "code_module",
        "code_presentation",
        "id_assessment",
        "assessment_type",
        "date",
        "weight",
    ],
    "studentInfo.csv": [
        "code_module",
        "code_presentation",
        "id_student",
        "gender",
        "region",
        "highest_education",
        "imd_band",
        "age_band",
        "num_of_prev_attempts",
        "studied_credits",
        "disability",
        "final_result",
    ],
    "studentRegistration.csv": [
        "code_module",
        "code_presentation",
        "id_student",
        "date_registration",
        "date_unregistration",
    ],
    "studentAssessment.csv": [
        "id_assessment",
        "id_student",
        "date_submitted",
        "is_banked",
        "score",
    ],
    "vle.csv": [
        "id_site",
        "code_module",
        "code_presentation",
        "activity_type",
        "week_from",
        "week_to",
    ],
    "studentVle.csv": [
        "code_module",
        "code_presentation",
        "id_student",
        "id_site",
        "date",
        "sum_click",
    ],
}

print("=== Source column validation ===")

for filename, expected in expected_columns.items():
    df = pd.read_csv(
        RAW_DATA_DIR / filename,
        nrows=0,
        keep_default_na=False
    )

    actual = df.columns.tolist()

    print(f"\n{filename}")
    print(f"Match: {actual == expected}")

    if actual != expected:
        print(f"Expected: {expected}")
        print(f"Actual:   {actual}")

=== Source column validation ===

courses.csv
Match: True

assessments.csv
Match: True

studentInfo.csv
Match: True

studentRegistration.csv
Match: True

studentAssessment.csv
Match: True

vle.csv
Match: True

studentVle.csv
Match: True


## Interpretation

Every file must return:

`Match: True`

If any file returns `False`, stop before PostgreSQL COPY and investigate the mismatch.

# Phase 2A — Validation Findings and RAW Design Decision

## Findings

The source validation produced the following decisions:

- `assessments.weight` contains real decimal values such as `7.5`, `12.5`, and `17.5`.
  Therefore, RAW PostgreSQL must use `NUMERIC`.

- `assessments.date` contains missing/source placeholder values.
  Therefore, it must remain nullable in RAW and be stored as `TEXT`.

- `studentAssessment.score` contains `?` placeholders.
  Although all non-missing numeric values are integer-valued, the RAW layer must preserve the source representation.
  Therefore, `score` will be stored as nullable `TEXT` in RAW.

- `studentRegistration.date_registration` contains missing/source placeholder values.
  Therefore, it must remain nullable in RAW and be stored as `TEXT`.

- `studentRegistration.date_unregistration` contains source `?` values.
  Therefore, it remains nullable `TEXT`.

- `studentInfo.imd_band` contains missing values.
  Therefore, it remains nullable `TEXT`.

- `vle.week_from` and `vle.week_to` contain source missing values.
  Therefore, both remain nullable `TEXT`.

- All seven CSV files matched the expected source column names and column order.

## RAW Design Principle

The RAW layer preserves source column names and source values while assigning validated PostgreSQL data types.

No cleaning, aggregation, duplicate removal, or business transformation is performed in RAW.

## Decision

The validated findings are sufficient to proceed to RAW table creation.

# Phase 2A — Gate Status

**Status:** Complete.

The source structure, candidate keys, RAW data types, source missing-value behavior, and CSV column order were validated before RAW table creation.

Phase 2B and Phase 2C have subsequently been completed in PostgreSQL. The next gate is **Phase 2D — `student_vle` duplicate characterization**.

No aggregation or duplicate-removal rule should be implemented until those duplicate groups are understood.